In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

PROJECT_DIR = Path(r"D:\AI-MSU")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR = PROCESSED_DIR / "_feature_temp"
TEMP_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 100_000

print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)

Raw data: D:\AI-MSU\data\raw
Processed data: D:\AI-MSU\data\processed


In [2]:
def find_file(filename):
    matches = list(RAW_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {RAW_DIR}")
    return matches[0]

source_files = {
    "logon": find_file("logon.csv"),
    "device": find_file("device.csv"),
    "file": find_file("file.csv"),
    "email": find_file("email.csv"),
    "http": find_file("http.csv")
}

for name, path in source_files.items():
    print(name, "->", path, f"({path.stat().st_size / (1024**3):.2f} GB)")

logon -> D:\AI-MSU\data\raw\logon.csv (0.05 GB)
device -> D:\AI-MSU\data\raw\device.csv (0.03 GB)
file -> D:\AI-MSU\data\raw\file.csv (0.18 GB)
email -> D:\AI-MSU\data\raw\email.csv (1.27 GB)
http -> D:\AI-MSU\data\raw\http.csv (13.54 GB)


In [3]:
def get_column(columns, candidates, required=True):
    lookup = {str(col).strip().lower(): col for col in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    if required:
        raise ValueError(f"Required column not found. Expected one of {candidates}; found {list(columns)}")
    return None


def aggregate_event_file(path, prefix, email_features=False):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    date_col = get_column(header, ["date", "timestamp", "datetime"])
    user_col = get_column(header, ["user", "username", "employee", "employee_name"])
    size_col = get_column(header, ["size", "email_size"], required=False) if email_features else None
    attach_col = get_column(header, ["attachments", "attachment_count"], required=False) if email_features else None
    usecols = list(dict.fromkeys([c for c in [date_col, user_col, size_col, attach_col] if c is not None]))
    temp_path = TEMP_DIR / f"{prefix}_partial.csv"
    if temp_path.exists():
        temp_path.unlink()
    first_write = True
    valid_rows = 0

    for chunk in pd.read_csv(path, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=True):
        chunk[date_col] = pd.to_datetime(chunk[date_col], errors="coerce")
        chunk = chunk.dropna(subset=[date_col, user_col])
        if chunk.empty:
            continue
        chunk["day"] = chunk[date_col].dt.normalize()
        chunk["after_hours"] = ((chunk[date_col].dt.hour < 8) | (chunk[date_col].dt.hour >= 18)).astype("int8")
        chunk["event_count"] = 1
        agg = {
            "event_count": "sum",
            "after_hours": "sum"
        }
        if size_col is not None:
            chunk[size_col] = pd.to_numeric(chunk[size_col], errors="coerce").fillna(0)
            agg[size_col] = "sum"
        if attach_col is not None:
            chunk[attach_col] = pd.to_numeric(chunk[attach_col], errors="coerce").fillna(0)
            agg[attach_col] = "sum"
        grouped = chunk.groupby([user_col, "day"], dropna=False).agg(agg).reset_index()
        rename = {
            user_col: "user",
            "event_count": f"{prefix}_event_count",
            "after_hours": f"{prefix}_after_hours"
        }
        if size_col is not None:
            rename[size_col] = "email_size"
        if attach_col is not None:
            rename[attach_col] = "email_attachments"
        grouped = grouped.rename(columns=rename)
        grouped.to_csv(temp_path, mode="a", header=first_write, index=False)
        first_write = False
        valid_rows += len(chunk)
        del chunk, grouped
        gc.collect()

    if first_write:
        raise ValueError(f"No valid rows were processed from {path}")

    partial = pd.read_csv(temp_path, parse_dates=["day"])
    value_cols = [col for col in partial.columns if col not in ["user", "day"]]
    daily = partial.groupby(["user", "day"], as_index=False)[value_cols].sum()
    output_path = PROCESSED_DIR / f"{prefix}_daily_features.csv"
    daily.to_csv(output_path, index=False)
    temp_path.unlink(missing_ok=True)
    del partial, daily
    gc.collect()
    print(f"Saved {prefix} daily features; valid source rows: {valid_rows:,}")
    return output_path

In [4]:
daily_paths = {}
for name in ["logon", "device", "file", "email", "http"]:
    daily_paths[name] = aggregate_event_file(
        source_files[name],
        name,
        email_features=(name == "email")
    )

print("\nCreated daily feature files:")
for name, path in daily_paths.items():
    print(name, path)

Saved logon daily features; valid source rows: 854,859
Saved device daily features; valid source rows: 405,380
Saved file daily features; valid source rows: 445,581
Saved email daily features; valid source rows: 2,629,979
Saved http daily features; valid source rows: 28,434,423

Created daily feature files:
logon D:\AI-MSU\data\processed\logon_daily_features.csv
device D:\AI-MSU\data\processed\device_daily_features.csv
file D:\AI-MSU\data\processed\file_daily_features.csv
email D:\AI-MSU\data\processed\email_daily_features.csv
http D:\AI-MSU\data\processed\http_daily_features.csv


In [5]:
daily_features = None
for name, path in daily_paths.items():
    part = pd.read_csv(path, parse_dates=["day"])
    if daily_features is None:
        daily_features = part
    else:
        daily_features = daily_features.merge(part, on=["user", "day"], how="outer")
    del part
    gc.collect()

daily_features = daily_features.fillna(0)
count_cols = [c for c in daily_features.columns if c not in ["user", "day"]]
daily_features[count_cols] = daily_features[count_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
daily_features = daily_features.sort_values(["user", "day"]).reset_index(drop=True)
daily_features.to_csv(PROCESSED_DIR / "daily_features.csv", index=False)

print("Shape:", daily_features.shape)
print("Date range:", daily_features["day"].min(), "to", daily_features["day"].max())
print("Users:", daily_features["user"].nunique())
display(daily_features.head())

Shape: (330452, 14)
Date range: 2010-01-02 00:00:00 to 2011-05-17 00:00:00
Users: 1000


,user,day,logon_event_count,logon_after_hours,device_event_count,device_after_hours,file_event_count,file_after_hours,email_event_count,email_after_hours,email_size,email_attachments,http_event_count,http_after_hours
0,AAE0190,2010-01-04,2,1,0.0,0.0,0.0,0.0,14.0,0.0,441328.0,4.0,143.0,7.0
1,AAE0190,2010-01-05,2,1,0.0,0.0,0.0,0.0,13.0,0.0,355552.0,2.0,143.0,1.0
2,AAE0190,2010-01-06,2,1,0.0,0.0,0.0,0.0,14.0,0.0,532647.0,12.0,143.0,3.0
3,AAE0190,2010-01-07,2,1,0.0,0.0,0.0,0.0,14.0,0.0,474631.0,9.0,143.0,8.0
4,AAE0190,2010-01-08,2,1,0.0,0.0,0.0,0.0,13.0,0.0,400637.0,10.0,143.0,3.0


In [6]:
users = sorted(daily_features["user"].dropna().unique())
all_days = pd.date_range(daily_features["day"].min(), daily_features["day"].max(), freq="D")
complete_index = pd.MultiIndex.from_product([users, all_days], names=["user", "day"])
complete_daily = daily_features.set_index(["user", "day"]).reindex(complete_index).reset_index()

feature_cols = [c for c in complete_daily.columns if c not in ["user", "day"]]
complete_daily[feature_cols] = complete_daily[feature_cols].fillna(0)
complete_daily = complete_daily.sort_values(["user", "day"]).reset_index(drop=True)
complete_daily.to_csv(PROCESSED_DIR / "complete_daily_features.csv", index=False)

print("Complete grid shape:", complete_daily.shape)
print("Missing values:", int(complete_daily.isna().sum().sum()))
print("Duplicate user-day rows:", int(complete_daily.duplicated(["user", "day"]).sum()))
print("Saved:", PROCESSED_DIR / "complete_daily_features.csv")
display(complete_daily.head())

Complete grid shape: (501000, 14)
Missing values: 0
Duplicate user-day rows: 0
Saved: D:\AI-MSU\data\processed\complete_daily_features.csv


,user,day,logon_event_count,logon_after_hours,device_event_count,device_after_hours,file_event_count,file_after_hours,email_event_count,email_after_hours,email_size,email_attachments,http_event_count,http_after_hours
0,AAE0190,2010-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AAE0190,2010-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AAE0190,2010-01-04,2.0,1.0,0.0,0.0,0.0,0.0,14.0,0.0,441328.0,4.0,143.0,7.0
3,AAE0190,2010-01-05,2.0,1.0,0.0,0.0,0.0,0.0,13.0,0.0,355552.0,2.0,143.0,1.0
4,AAE0190,2010-01-06,2.0,1.0,0.0,0.0,0.0,0.0,14.0,0.0,532647.0,12.0,143.0,3.0
